# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, overview, and analyze the dataset "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the FAIR² dataset Croissant metadata and prepare for browsing and analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access the full metadata object
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}")

## 2. Data Overview
Let's examine the available record sets and their corresponding fields, listing all unique `@id`s (identifiers).

> **Note:** All references to record sets and fields use their `@id` fields, as required for Croissant-based workflows.

In [ ]:
# List all record sets with their `@id` and included fields
record_set_objs = list(dataset.record_sets)
if not record_set_objs:
    print("No record sets are defined in the metadata. If this occurs, the source may use top-level data instead.")
else:
    for rs in record_set_objs:
        print(f"RecordSet name: {rs.name}\n@id: {rs.id}")
        print("  Fields:")
        for f in rs.fields:
            print(f"   - {f.name} (@id: {f.id}) | type: {f.data_type}")
        print('-'*40)
        # Try showing first 1-2 records
        try:
            sample_records = list(dataset.records(record_set=rs.id))
            print(f"Sample record: {sample_records[0] if sample_records else 'None'}\n")
        except Exception as e:
            print(f"Could not load records for {rs.id}: {str(e)}\n")
        print('\n')

### Determine available record sets for loading
For this dataset, let's extract the list of defined record set `@id`s. If none exist, we'll attempt to infer from available distributions.

In [ ]:
# Get a list of all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Record Set @ids:", record_set_ids)

if not record_set_ids:
    # If none found, perhaps fall back and check for distributions
    ds_json = dataset.metadata.to_json() if hasattr(dataset.metadata, 'to_json') else dataset.metadata
    distributions = ds_json.get('distribution', [])
    print(f"No Croissant record sets. The available distributions (@id): {[d.get('@id') for d in distributions]}")

## 3. Data Extraction
Load records from each record set into pandas DataFrames, referencing each by its `@id`. If there are no record sets, we'll note this accordingly.


In [ ]:
# Load data from all record sets (referenced by @id)
dataframes = {}
if record_set_ids:
    for record_set_id in record_set_ids:
        print(f"\nLoading records for record set: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print("Columns:", df.columns.tolist())
            display(df.head())
        else:
            print(f"No records loaded for record set {record_set_id}.")
else:
    print("No Croissant record sets defined. Please check source documentation for the data structure.")

## 4. Exploratory Data Analysis (EDA)
Let's perform basic data processing and summarization for one of the loaded record sets.

1. **Filter** records for a specific numeric field.
2. **Normalize** this numeric field.
3. **Group** by a categorical field, if available.

*You must replace `<field_id>` with the true field `@id` as listed above, e.g., `'field:log_likelihood'`.*

In [ ]:
# Example: EDA for the first loaded record set (customize as needed)
if dataframes:
    # Pick the first record set for illustration
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"Operating on record set: {rs_id}, columns: {list(df.columns)}\n")

    # List candidate numeric and categorical columns by their `@id`
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    group_candidates = [c for c in df.columns if pd.api.types.is_object_dtype(df[c])]
    print(f"Numeric candidates: {numeric_candidates}")
    print(f"Group-by candidates: {group_candidates}")

    # For demo, pick the first numeric and first group field (update for your dataset):
    numeric_field = numeric_candidates[0] if numeric_candidates else None
    group_field = group_candidates[0] if group_candidates else None

    if numeric_field:
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field]).all() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field}@id > {threshold} (mean):\n", filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No numeric fields found to process.")

    # Group by group_field if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field}:\n", grouped_df.head())
    else:
        print("No available group field for grouping.")
else:
    print("No dataframes loaded to perform EDA.")

## 5. Visualization

Visualize the distribution of a selected numeric field, and (if possible) relationships with a grouping column.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if dataframes and numeric_field:
    # Histogram of the numeric field
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} (@id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group_field if available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"Boxplot of {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load the FAIR² dataset and explored its metadata, structure, and sample records using reproducible Data Science practices with structured `@id` referencing. We demonstrated data loading, basic EDA, normalization, grouping, and visualized data distributions. Refer to the dataset's Croissant schema for full documentation of field meanings and detailed provenance.

**Next steps:** Perform advanced statistical analysis or modeling, select specific variables for in-depth policy or scientific insights, and review the Croissant metadata for best practices in FAIR data workflows.